In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/test_queries.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/sample_submission.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/documents.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/dataset-metadata.json
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/train_queries.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/qrels_train.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/baseline_submission.csv


In [2]:
import pandas as pd
import numpy as np
df = pd.read_csv("/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/documents.csv", index_col ="document_id")
test = pd.read_csv("/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/test_queries.csv")
df.head()
test.head()

,query_id,query
0,1001,How do I cope with drought and erratic rainfal...
1,1002,How can I adapt my farming to drought and erra...
2,1003,How does drought and erratic rainfall affect m...
3,1004,What is the risk of drought and erratic rainfa...
4,1005,How do I cope with heat stress on my farm?


In [3]:
# My plan is to use hybird method which involves
# bm25 + dense retrival + reranker
def create_search_content(row):
    title = str(row.get("title", " ")).strip()
    text = str(row.get("text", " ")).strip()
    source = str(row.get("source", " ")).strip()
    crop = str(row.get("crop", " ")).strip()
    country = str(row.get("country", " ")).strip()
    source_url = str(row.get("source_url", " ")).strip()
    
    return f"Crop {crop} | Country {country} | Title {title} | Text {text} | Source {source}"
df_upd=pd.DataFrame()
df["search_text"] = df.apply(create_search_content, axis = 1)
docs_ids = df.index.to_list()
corpus_texts = df['search_text'].tolist()

In [4]:
# 2. Build Training Pairs (Positives + Hard Negatives)
# ----------------------------------------------------
train_queries_df = pd.read_csv('/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/train_queries.csv')
qrels_train_df = pd.read_csv('/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/qrels_train.csv')

In [5]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared.")

GPU memory cleared.


In [6]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModel,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

# ----------------------------------------------------
# 1. Multi-GPU Device Setup & Model Initialization
# ----------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_gpus = torch.cuda.device_count()
print(f"Running on: {device} | Total GPUs available: {num_gpus}")

model_name = "BAAI/bge-base-en-v1.5"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_encoder = AutoModel.from_pretrained(model_name)

# Mean pooling + normalization module
class BiEncoderModel(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # Mean pooling with attention mask
        token_embeddings = outputs.last_hidden_state
        input_mask_expanded = (
            attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        )
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        pooled = sum_embeddings / sum_mask
        return F.normalize(pooled, p=2, dim=1)

model = BiEncoderModel(base_encoder).to(device)

# Distribute across both GPUs
if num_gpus > 1:
    print(f"Distributing encoder across {num_gpus} GPUs via DataParallel.")
    model = nn.DataParallel(model)

# ----------------------------------------------------
# 2. Build Training Triplets (Positives + Hard Negatives)
# ----------------------------------------------------
train_queries_df = pd.read_csv('/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/train_queries.csv')
qrels_train_df = pd.read_csv('/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/qrels_train.csv')

tq_id_col = 'QueryId' if 'QueryId' in train_queries_df.columns else train_queries_df.columns[0]
tq_text_col = 'Query' if 'Query' in train_queries_df.columns else train_queries_df.columns[1]

qrels_qid_col = 'QueryId' if 'QueryId' in qrels_train_df.columns else qrels_train_df.columns[0]
qrels_did_col = 'DocumentId' if 'DocumentId' in qrels_train_df.columns else qrels_train_df.columns[1]

query_dict = dict(zip(train_queries_df[tq_id_col], train_queries_df[tq_text_col]))
doc_dict = dict(zip(docs_ids, corpus_texts))

BGE_PREFIX = "Represent this sentence for searching relevant passages: "

triplet_queries = []
triplet_positives = []
triplet_negatives = []

for q_id, group in qrels_train_df.groupby(qrels_qid_col):
    if q_id not in query_dict:
        continue

    # Queries require the asymmetric instruction prefix for BGE
    q_text = BGE_PREFIX + str(query_dict[q_id]).strip()

    pos_docs = group[group['relevance'] >= 2][qrels_did_col].tolist()
    neg_docs = group[group['relevance'] == 0][qrels_did_col].tolist()

    if not pos_docs:
        continue

    for pos_id in pos_docs:
        if pos_id not in doc_dict:
            continue
        p_text = doc_dict[pos_id]

        if neg_docs:
            for neg_id in neg_docs:
                if neg_id in doc_dict:
                    triplet_queries.append(q_text)
                    triplet_positives.append(p_text)
                    triplet_negatives.append(doc_dict[neg_id])
        else:
            # Fallback: self-negative (will still be contrasted against in-batch negatives)
            triplet_queries.append(q_text)
            triplet_positives.append(p_text)
            triplet_negatives.append(p_text)

print(f"Total training triplets generated: {len(triplet_queries):,}")

# ----------------------------------------------------
# 3. Dataset & Multi-GPU DataLoader
# ----------------------------------------------------
class TripletDataset(Dataset):
    def __init__(self, queries, positives, negatives, tokenizer, max_length=256):
        self.queries = queries
        self.positives = positives
        self.negatives = negatives
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.queries)

    def __getitem__(self, idx):
        enc_q = self.tokenizer(
            self.queries[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        enc_p = self.tokenizer(
            self.positives[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        enc_n = self.tokenizer(
            self.negatives[idx],
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "q_input_ids": enc_q["input_ids"].squeeze(0),
            "q_attention_mask": enc_q["attention_mask"].squeeze(0),
            "p_input_ids": enc_p["input_ids"].squeeze(0),
            "p_attention_mask": enc_p["attention_mask"].squeeze(0),
            "n_input_ids": enc_n["input_ids"].squeeze(0),
            "n_attention_mask": enc_n["attention_mask"].squeeze(0),
        }

train_dataset = TripletDataset(
    triplet_queries,
    triplet_positives,
    triplet_negatives,
    tokenizer,
    max_length=256
)

# 32 batch size per GPU -> effective batch size 32 on 2x T4
per_gpu_batch = 16
effective_batch_size = per_gpu_batch * max(1, num_gpus)

train_loader = DataLoader(
    train_dataset,
    batch_size=effective_batch_size,
    shuffle=True,
    drop_last=True,  # Crucial for stable in-batch contrastive sizing
    num_workers=2,
    pin_memory=True
)

# ----------------------------------------------------
# 4. Multi-GPU Training Loop (InfoNCE / MNRL)
# ----------------------------------------------------
# Scale factor (temperature tau = 0.05 -> 1/0.05 = 20.0) standard for cosine contrastive loss
SCALE = 20.0
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
epochs = 3

total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps
)
criterion = nn.CrossEntropyLoss()

model.train()
print(f"Beginning training across {num_gpus} GPUs (Effective batch: {effective_batch_size})...")

for epoch in range(epochs):
    running_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()

        q_ids = batch["q_input_ids"].to(device)
        q_mask = batch["q_attention_mask"].to(device)
        p_ids = batch["p_input_ids"].to(device)
        p_mask = batch["p_attention_mask"].to(device)
        n_ids = batch["n_input_ids"].to(device)
        n_mask = batch["n_attention_mask"].to(device)

        # Forward passes split across both GPUs via DataParallel
        q_emb = model(q_ids, q_mask)  # (Batch, Dim)
        p_emb = model(p_ids, p_mask)  # (Batch, Dim)
        n_emb = model(n_ids, n_mask)  # (Batch, Dim)

        # Candidates pool = [All Positives in batch; All Negatives in batch]
        candidates = torch.cat([p_emb, n_emb], dim=0)  # (2 * Batch, Dim)

        # Cosine similarity matrix scaled by temperature
        sim_matrix = torch.matmul(q_emb, candidates.T) * SCALE  # (Batch, 2 * Batch)

        # Target: Anchor_i matches Positive_i (indices 0 to Batch-1)
        targets = torch.arange(q_emb.size(0), device=device)

        loss = criterion(sim_matrix, targets)
        loss.backward()

        optimizer.step()
        scheduler.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{epochs} - Contrastive Loss: {avg_loss:.4f}")

# ----------------------------------------------------
# 5. Save Model in Standard Hugging Face / SBERT Format
# ----------------------------------------------------
save_dir = "./fine_tuned_bge_base_agri"
os.makedirs(save_dir, exist_ok=True)

# Unwrap DataParallel before export
raw_model = model.module if isinstance(model, nn.DataParallel) else model
raw_model.encoder.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"Fine-tuning complete! Model saved cleanly to '{save_dir}'.")

Running on: cuda | Total GPUs available: 2


config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Distributing encoder across 2 GPUs via DataParallel.
Total training triplets generated: 6,108
Beginning training across 2 GPUs (Effective batch: 32)...
Epoch 1/3 - Contrastive Loss: 0.5273
Epoch 2/3 - Contrastive Loss: 0.3614
Epoch 3/3 - Contrastive Loss: 0.3582


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuning complete! Model saved cleanly to './fine_tuned_bge_base_agri'.
